In [1]:
import os 
import cv2
import random
import numpy as np
import tensorflow as tf

from pathlib import Path
from tqdm.auto import tqdm

In [2]:
DATASET_DIR = Path("/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23")

ORIGINAL_DIR = DATASET_DIR/"original"

FAKE_DIRS = [
    DATASET_DIR/"DeepFakeDetection",
    DATASET_DIR/"Deepfakes",
    DATASET_DIR/"Face2Face",
    DATASET_DIR/"FaceShifter",
    DATASET_DIR/"FaceSwap",
    DATASET_DIR/"NeuralTextures"
]

print("Dataset exists: ", DATASET_DIR.exists())
print("Original exist: ", ORIGINAL_DIR.exists())

for folder in FAKE_DIRS:
    print(folder.name, ":", folder.exists())

Dataset exists:  True
Original exist:  True
DeepFakeDetection : True
Deepfakes : True
Face2Face : True
FaceShifter : True
FaceSwap : True
NeuralTextures : True


In [3]:
SEED = 42

FRAMES_PER_VIDEO = 10

IMAGE_SIZE = (299, 299)

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [4]:
VIDEO_EXTENSIONS = {
    ".mp4",
    ".avi",
    ".mov",
    ".mkv"
}

def find_videos(folder):
    videos = []

    for path in Path(folder).rglob("*"):
        if path.suffix.lower() in VIDEO_EXTENSIONS:
            videos.append(path)

    return videos


real_videos = find_videos(ORIGINAL_DIR)

fake_videos = []

for folder in FAKE_DIRS:
    videos = find_videos(folder)
    fake_videos.extend(videos)

print("Real videos : ", len(real_videos))
print("Fake videos : ", len(fake_videos))

Real videos :  1000
Fake videos :  6000


In [5]:
def split_videos(videos):
    videos = list(videos)
    
    random.Random(SEED).shuffle(videos)

    total = len(videos)

    train_end = int(total * TRAIN_RATIO)
    val_end = int(total * (TRAIN_RATIO + VAL_RATIO))

    train_videos = videos[:train_end]
    val_videos = videos[train_end:val_end]
    test_videos = videos[val_end:]

    return train_videos, val_videos, test_videos
    

In [6]:
real_train, real_val, real_test = split_videos(real_videos)
fake_train, fake_val, fake_test = split_videos(fake_videos)

In [7]:
print("REAL")
print("Train: ", len(real_train))
print("Validation: ", len(real_val))
print("Test: ", len(real_test))

print()

print("FAKE")
print("Train: ", len(fake_train))
print("Validation: ", len(fake_val))
print("Test: ", len(fake_test))

REAL
Train:  700
Validation:  150
Test:  150

FAKE
Train:  4200
Validation:  900
Test:  900


In [8]:
import zipfile
from concurrent.futures import ThreadPoolExecutor

OUT_DIR = Path("/kaggle/working/deepfake_frames")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESIZE = None       
JPEG_QUALITY = 95
WORKERS = 8


def video_uid(video_path):
    rel = Path(video_path).relative_to(DATASET_DIR).with_suffix("")
    return "_".join(rel.parts)


def read_frames(job):
    video_path, label = job
    uid = video_uid(video_path)
    out = []

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print("Could not open:", video_path)
        return out

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return out

    idxs = np.linspace(0, total - 1, FRAMES_PER_VIDEO, dtype=int)

    for i, idx in enumerate(idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            continue
        if RESIZE:
            frame = cv2.resize(frame, RESIZE)
        ok, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        if ok:
            out.append((f"{label}/{uid}/frame_{i:03d}.jpg", buf.tobytes()))

    cap.release()
    return out


def build_zip(split, real_list, fake_list):
    jobs = [(v, "real") for v in real_list] + [(v, "fake") for v in fake_list]
    zip_path = OUT_DIR / f"{split}.zip"
    n = 0

   
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as zf, \
         ThreadPoolExecutor(max_workers=WORKERS) as ex:
        for frames in tqdm(ex.map(read_frames, jobs), total=len(jobs), desc=split):
            for name, data in frames:
                zf.writestr(name, data)
                n += 1

    size_gb = zip_path.stat().st_size / 1e9
    print(f"{split}: {n} frames -> {zip_path.name} ({size_gb:.2f} GB)")



build_zip("val",   real_val,   fake_val)
build_zip("test",  real_test,  fake_test)
build_zip("train", real_train, fake_train)

val:   0%|          | 0/1050 [00:00<?, ?it/s]

val: 10500 frames -> val.zip (1.80 GB)


test:   0%|          | 0/1050 [00:00<?, ?it/s]

test: 10500 frames -> test.zip (1.81 GB)


train:   0%|          | 0/4900 [00:01<?, ?it/s]

train: 49000 frames -> train.zip (8.28 GB)


In [9]:
for split in ["train", "val", "test"]:
    with zipfile.ZipFile(OUT_DIR / f"{split}.zip") as zf:
        names = zf.namelist()
    r = sum(n.startswith("real/") for n in names)
    f = sum(n.startswith("fake/") for n in names)
    print(f"{split}: real={r}  fake={f}")

train: real=7000  fake=42000
val: real=1500  fake=9000
test: real=1500  fake=9000
